<a href="https://colab.research.google.com/github/RadhikaDeshpande1010/PySpark-RDD-Analytics-Starbucks-Global-Stores/blob/main/starbucks_rdd_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local") \
    .appName("Starbucks_RDD_Analysis") \
    .getOrCreate()

print(spark)

In [2]:
sc = spark.sparkContext
print(sc)

<SparkContext master=local appName=Starbucks_RDD_Analysis>


# ☕ PySpark RDD Analytics — Starbucks Global Stores Dataset

**Domain:** Retail Store Analytics | **Engine:** Apache Spark (RDD API) | **Language:** Python 3

---

## 📋 Dataset Schema

| Field | Index | Type | Description |
|---|---|---|---|
| `Brand` | 0 | str | Always `Starbucks` |
| `Store Number` | 1 | str | Unique store identifier |
| `Store Name` | 2 | str | Name of the store |
| `Ownership Type` | 3 | str | `Company Owned` / `Licensed` / `Joint Venture` / `Franchise` |
| `Street Address` | 4 | str | Full street address |
| `City` | 5 | str | City name |
| `State/Province` | 6 | str | State or province code |
| `Country` | 7 | str | ISO 2-letter country code |
| `Postcode` | 8 | str | Postal / ZIP code |
| `Phone Number` | 9 | str | Store phone number |
| `Timezone` | 10 | str | IANA timezone string |
| `Longitude` | 11 | float | Longitude coordinate |
| `Latitude` | 12 | float | Latitude coordinate |

**Sample record:**
`Starbucks,47370-257954,"Meritxell, 96",Licensed,"Av. Meritxell, 96",Andorra la Vella,7,AD,AD500,376818720,GMT+1:00 Europe/Andorra,1.53,42.51`

---

## 📂 Exercises Overview

| Q# | Question |
|---|---|
| Q1 | Number of Starbucks shops per country |
| Q2 | Total number of Licensed shops worldwide |
| Q3 | How many shops are in the Middle East region? |
| Q4 | How many shops are in the city of Wien? |
| Q5 | Print all unique postcodes in the dataset |
| Q6 | How many shops are on Airport Road? |
| Q7 | Which country has the highest number of Starbucks stores? |
| Q8 | How many shops are Company Owned worldwide? |
| Q9 | List all unique ownership types in the dataset |
| Q10 | How many stores are available in the US? |
| Q11 | Print the top 5 countries with the most Starbucks stores |
| Q12 | How many stores operate in each timezone? |

In [4]:
import csv

# Load raw CSV, strip header, parse each row with csv.reader to handle quoted fields
raw_rdd    = sc.textFile("/content/sample_data/starbucks_data.csv")
header     = raw_rdd.first()

data_rdd = raw_rdd \
    .filter(lambda row: row != header) \
    .map(lambda row: next(csv.reader([row])))

print(f"Total stores: {data_rdd.count()}")
data_rdd.take(2)

Total stores: 25601


[['Starbucks',
  '47370-257954',
  'Meritxell, 96',
  'Licensed',
  'Av. Meritxell, 96',
  'Andorra la Vella',
  '7',
  'AD',
  'AD500',
  '376818720',
  'GMT+1:00 Europe/Andorra',
  '1.53',
  '42.51'],
 ['Starbucks',
  '22331-212325',
  'Ajman Drive Thru',
  'Licensed',
  '1 Street 69, Al Jarf',
  'Ajman',
  'AJ',
  'AE',
  '',
  '',
  'GMT+04:00 Asia/Dubai',
  '55.47',
  '25.42']]

---
## 🌍 Store Distribution & Ownership Analysis

Exercises 1–6: country counts, ownership filters, region filters, city-level queries.

In [5]:
# Q1 — Number of Starbucks shops per country
# col[7] = Country (ISO 2-letter code)
shops_count_per_country_rdd = data_rdd \
    .filter(lambda col: len(col) > 7) \
    .map(lambda col: (col[7], 1)) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False)

print(shops_count_per_country_rdd.collect())

[('US', 13608), ('CN', 2734), ('CA', 1468), ('JP', 1237), ('KR', 993), ('GB', 900), ('MX', 579), ('TW', 394), ('TR', 326), ('PH', 298), ('TH', 289), ('ID', 268), ('MY', 234), ('DE', 160), ('AE', 144), ('FR', 132), ('SG', 130), ('RU', 109), ('AR', 108), ('KW', 106), ('BR', 102), ('SA', 102), ('ES', 101), ('CL', 96), ('PE', 89), ('IN', 88), ('IE', 73), ('CH', 61), ('NL', 59), ('PL', 53), ('EG', 31), ('LB', 29), ('CZ', 28), ('GR', 28), ('RO', 27), ('VN', 25), ('NZ', 24), ('PR', 24), ('AU', 22), ('BH', 21), ('DK', 21), ('BE', 19), ('AT', 18), ('QA', 18), ('SE', 18), ('JO', 17), ('NO', 17), ('HU', 16), ('OM', 12), ('CO', 11), ('CR', 11), ('PT', 11), ('SV', 11), ('BS', 10), ('CY', 10), ('MA', 9), ('FI', 8), ('KZ', 8), ('GT', 7), ('BG', 5), ('BN', 5), ('PA', 5), ('AZ', 4), ('BO', 4), ('KH', 4), ('AW', 3), ('CW', 3), ('SK', 3), ('TT', 3), ('ZA', 3), ('LU', 2), ('MC', 2), ('AD', 1)]


In [6]:
# Q2 — Total number of Licensed shops worldwide
# col[3] = Ownership Type
licensed_shops_count = data_rdd \
    .filter(lambda col: len(col) > 3) \
    .filter(lambda col: col[3] == 'Licensed') \
    .count()

print(f"Total Licensed shops worldwide: {licensed_shops_count}")

Total Licensed shops worldwide: 9375


In [7]:
# Q3 — How many shops are in the Middle East region?
# Middle East ISO country codes
middle_east_country_codes = ['AE', 'SA', 'QA', 'KW', 'BH', 'OM', 'JO', 'LB']

middle_east_shops_count = data_rdd \
    .filter(lambda col: len(col) > 7) \
    .filter(lambda col: col[7] in middle_east_country_codes) \
    .count()

print(f"Starbucks shops in Middle East: {middle_east_shops_count}")

Starbucks shops in Middle East: 449


In [8]:
# Q4 — How many shops are in the city of Wien (Vienna, Austria)?
# col[5] = City
wien_city_shops_count = data_rdd \
    .filter(lambda col: len(col) > 5) \
    .filter(lambda col: col[5] == 'Wien') \
    .count()

print(f"Starbucks shops in Wien: {wien_city_shops_count}")

Starbucks shops in Wien: 13


In [9]:
# Q5 — Print all unique postcodes present in the dataset
# col[8] = Postcode
unique_postcodes_rdd = data_rdd \
    .filter(lambda col: len(col) > 8) \
    .map(lambda col: col[8].strip()) \
    .filter(lambda postcode: postcode != '') \
    .distinct() \
    .sortBy(lambda x: x)

print(f"Total unique postcodes: {unique_postcodes_rdd.count()}")
print(unique_postcodes_rdd.collect())

Total unique postcodes: 18886
['0', '00-023', '00-024', '00-113', '00-127', '00-144', '00-240', '00-357', '00-499', '00-635', '00-697', '004-0866', '004-0873', '005-0842', '007-0802', '010-0003', '010-1413', '010-8530', '010-8543', '01238-000', '01307-001', '01310-100', '01311-000', '01311-200', '01327-001', '014-0033', '01413-000', '01414-000', '01418-100', '01448-900', '02 326', '02-515', '02-697', '02-972', '020-0024', '020-0034', '020-0148', '020-0857', '020-0866', '02089-900', '02240-000', '024-0072', '03-287', '03-734', '030-0111', '03066-030', '031-0011', '031-0072', '03126-000', '03164-000', '03306-010', '03342-000', '036-8207', '037-0004', '038-0006', '038-8555', '039-2112', '04-', '04-175', '040-0053', '04001-080', '04004-030', '04029-902', '04036-100', '041-0802', '04151-100', '04523-012', '04532-081', '04534-002', '04543-011', '04546-001', '04547-002', '04571-010', '04583-903', '04707-970', '04795-000', '04795-100', '05005-000', '05069-010', '053-0053', '05477-000', '05620-

In [10]:
# Q6 — How many shops are on Airport Road? → (store_number, store_name, address, city, country)
# col[4] = Street Address
airport_road_shops_rdd = data_rdd \
    .filter(lambda col: len(col) > 7) \
    .filter(lambda col: 'Airport Road' in col[4]) \
    .map(lambda col: (col[1], col[2], col[4], col[5], col[7]))

print(f"Shops on Airport Road: {airport_road_shops_rdd.count()}")
print(airport_road_shops_rdd.collect())

Shops on Airport Road: 12
[('29539-254261', 'MUshrif Mall 1', '25th st, Airport Road', 'Abu Dhabi', 'AE'), ('28810-251346', 'ADNOC Mushrif', 'Airport Road', 'Abu Dhabi', 'AE'), ('23973-234507', 'Dubai Airport T1 Landside Departure', 'Airport Road, Dubai Airport, Terminal 1', 'Dubai', 'AE'), ('34217-27108', 'American University of Sharjah', 'Airport Road, Universities Complex', 'Sharjah', 'AE'), ('75779-94250', 'YYC Calg N Processor, Retail Hall', '2000 Airport Road', 'Calgary', 'CA'), ('70002-120504', 'YYC Gate a16', '2000 Airport Road NE, Westbrook Mall', 'Calgary', 'CA'), ('29386-253567', 'Sama Mall', 'Airport Road, Khaitan', 'Khaitan', 'KW'), ('21325-210925', 'Business Gate', 'Airport Road, Qurtubah District', 'Riyadh', 'SA'), ('21739-212544', 'Palomar Airport Rd & Armada', '965 Palomar Airport Road, 106', 'Carlsbad', 'US'), ('8685-94424', 'Rifle-I-70 & CO Rd 346', '900 Airport Road, 1', 'Rifle', 'US'), ('75957-105038', 'OGG Baggage Claim (Maui)', '1 Kahului Airport Road Unit 9', 'K

---
## 📊 Rankings, Ownership Types & Regional Intelligence

Exercises 7–12: top-N countries, Company Owned count, distinct types, US stores, timezone distribution.

In [11]:
# Q7 — Which country has the highest number of Starbucks stores?
# col[7] = Country
country_with_most_stores = data_rdd \
    .filter(lambda col: len(col) > 7) \
    .map(lambda col: (col[7], 1)) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False) \
    .first()

print(f"Country with most stores → Code: {country_with_most_stores[0]}, "
      f"Count: {country_with_most_stores[1]}")

Country with most stores → Code: US, Count: 13608


In [12]:
# Q8 — How many shops are Company Owned worldwide?
# col[3] = Ownership Type
company_owned_shops_count = data_rdd \
    .filter(lambda col: len(col) > 3) \
    .filter(lambda col: col[3] == 'Company Owned') \
    .count()

print(f"Total Company Owned shops worldwide: {company_owned_shops_count}")

Total Company Owned shops worldwide: 11932


In [13]:
# Q9 — List all unique ownership types present in the dataset
# col[3] = Ownership Type
unique_ownership_types_rdd = data_rdd \
    .filter(lambda col: len(col) > 3) \
    .map(lambda col: col[3].strip()) \
    .distinct() \
    .sortBy(lambda x: x)

print("Unique ownership types:")
print(unique_ownership_types_rdd.collect())

Unique ownership types:
['Company Owned', 'Franchise', 'Joint Venture', 'Licensed']


In [14]:
# Q10 — How many stores are available in the United States?
# col[7] = Country
us_stores_count = data_rdd \
    .filter(lambda col: len(col) > 7) \
    .filter(lambda col: col[7] == 'US') \
    .count()

print(f"Total Starbucks stores in US: {us_stores_count}")

Total Starbucks stores in US: 13608


In [15]:
# Q11 — Top 5 countries with the most Starbucks stores
# col[7] = Country
top_5_countries_by_store_count = data_rdd \
    .filter(lambda col: len(col) > 7) \
    .map(lambda col: (col[7], 1)) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(5)

print("Top 5 countries by store count:")
for rank, (country, count) in enumerate(top_5_countries_by_store_count, 1):
    print(f"  {rank}. {country}: {count} stores")

Top 5 countries by store count:
  1. US: 13608 stores
  2. CN: 2734 stores
  3. CA: 1468 stores
  4. JP: 1237 stores
  5. KR: 993 stores


In [16]:
# Q12 — How many stores operate in each timezone?
# col[10] = Timezone
stores_per_timezone_rdd = data_rdd \
    .filter(lambda col: len(col) > 10) \
    .filter(lambda col: col[10].strip() != '') \
    .map(lambda col: (col[10].strip(), 1)) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False)

print(f"Total unique timezones: {stores_per_timezone_rdd.count()}")
print(stores_per_timezone_rdd.collect())

Total unique timezones: 101
[('GMT-05:00 America/New_York', 4889), ('GMT-08:00 America/Los_Angeles', 4194), ('GMT-06:00 America/Chicago', 2901), ('GMT+08:00 Asia/Beijing', 2731), ('GMT+09:00 Asia/Tokyo', 1237), ('GMT+09:00 Asia/Seoul', 993), ('GMT+0:00 Europe/London', 900), ('GMT-07:00 America/Denver', 804), ('GMT-06:00 America/Mexico_City', 531), ('GMT-05:00 America/Toronto', 518), ('GMT+000000 America/Phoenix', 487), ('GMT-08:00 America/Vancouver', 407), ('GMT+08:00 Asia/Taipei', 393), ('GMT+07:00 Asia/Jakarta', 316), ('GMT+08:00 Asia/Manila', 298), ('GMT-07:00 America/Edmonton', 289), ('GMT+07:00 Asia/Bangkok', 289), ('GMT+08:00 Asia/Kuala_Lumpur', 282), ('GMT+2:00 Asia/Istanbul', 231), ('GMT-05:00 America/Indianapolis', 191), ('GMT+1:00 Europe/Berlin', 162), ('GMT+1:00 Europe/Paris', 136), ('GMT+04:00 Asia/Dubai', 134), ('GMT-03:00 America/Argentina/Bu', 108), ('GMT+000000 Asia/Kuwait', 106), ('GMT-03:00 America/Sao_Paulo', 102), ('GMT+3:00 Europe/Moscow', 102), ('GMT+03:00 Asia/Ri